In [1]:
!pip install -q \
transformers \
datasets \
accelerate \
peft \
trl \
sentencepiece \
protobuf \
huggingface_hub

In [2]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        i,
        torch.cuda.get_device_name(i),
        round(
            torch.cuda.get_device_properties(i).total_memory / 1024**3,
            1
        ),
        "GB"
    )

True
1
0 NVIDIA GeForce RTX 5090 31.4 GB


In [8]:
!pip install datasets
import sys
!{sys.executable} -m pip install datasets transformers peft accelerate trl sentencepiece

  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached trl-1.6.0-py3-none-any.whl.metadata (11 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached xxhash-3.7.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached aiohttp-3.14.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadat

In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "opencsg/PR_review_deepseek"
)["train"]

print("Samples:", len(dataset))
print(dataset.column_names)
print(dataset[0])    

README.md:   0%|          | 0.00/270 [00:00<?, ?B/s]

all_py_review_deepseek_28w_prompt_respon(…):   0%|          | 0.00/460M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24821 [00:00<?, ? examples/s]

Samples: 24821
['instruction', 'output', 'prompt_len', 'response_len', 'total_len']
{'instruction': 'Here is a python code snippet (Some lines may be omitted):\n\n```python\n\nimport sys\nimport threading\nimport signal\nimport array\nimport queue\nimport time\nimport os\nfrom os import getpid\n\nfrom traceback import format_exc\n\nfrom . import connection\nfrom .context import reduction, get_spawning_popen, ProcessError\nfrom . import pool\nfrom . import process\nfrom . import util\nfrom . import get_context\ntry:\n    from . import shared_memory\n    HAS_SHMEM = True\n    __all__.append(SharedMemoryManager)\nexcept ImportError:\n    HAS_SHMEM = False\n\n#\n# Register some things for pickling\n#\n\ndef reduce_array(a):\n    return array.array, (a.typecode, a.tobytes())\nreduction.register(array.array, reduce_array)\n\nview_types = [type(getattr({}, name)()) for name in (\'items\',\'keys\',\'values\')]\nif view_types[0] is not list:       # only needed in Py3.0\n    def rebuild_as_list

In [10]:
dataset.column_names

['instruction', 'output', 'prompt_len', 'response_len', 'total_len']

In [11]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [12]:
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [14]:
MAX_LENGTH = 1024

def format_sample(example):
    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {
        "text": text
    }

dataset = dataset.map(format_sample)

print(dataset[0]["text"][:1000])

Map:   0%|          | 0/24821 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Here is a python code snippet (Some lines may be omitted):

```python

import sys
import threading
import signal
import array
import queue
import time
import os
from os import getpid

from traceback import format_exc

from . import connection
from .context import reduction, get_spawning_popen, ProcessError
from . import pool
from . import process
from . import util
from . import get_context
try:
    from . import shared_memory
    HAS_SHMEM = True
    __all__.append(SharedMemoryManager)
except ImportError:
    HAS_SHMEM = False

#
# Register some things for pickling
#

def reduce_array(a):
    return array.array, (a.typecode, a.tobytes())
reduction.register(array.array, reduce_array)

view_types = [type(getattr({}, name)()) for name in ('items','keys','values')]
if view_types[0] is not list:       # only needed in Py3.0
    def rebuild_as_list(obj):
        return list, (l

In [42]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/24821 [00:00<?, ? examples/s]

In [43]:
from transformers import default_data_collator

data_collator = default_data_collator

In [44]:
len(tokenized_dataset)

24821

In [45]:
tokenized_dataset[0].keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [46]:
import numpy as np

lengths = [
    len(x["input_ids"])
    for x in tokenized_dataset.select(range(1000))
]

print("Average:", np.mean(lengths))
print("Max:", np.max(lengths))
print("Min:", np.min(lengths))

Average: 1024.0
Max: 1024
Min: 1024


In [47]:
for i in range(10):
    sample = tokenized_dataset[i]

    print(
        len(sample["input_ids"]),
        len(sample["attention_mask"]),
        len(sample["labels"])
    )

1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024
1024 1024 1024


In [48]:
training_args = TrainingArguments(
    output_dir="./pr-review-qwen",

    num_train_epochs=1,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,
    weight_decay=0.01,

    bf16=True,

    logging_steps=25,

    save_strategy="epoch",

    report_to="none",

    gradient_checkpointing=True,
)

In [49]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [50]:
import torch

print(
    f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
)
print(
    f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

Allocated: 3.28 GB
Reserved: 21.51 GB


In [51]:
trainer.train()

Step,Training Loss
25,1.087636
50,1.163007
75,1.112233
100,1.167953
125,1.145533
150,1.180122
175,1.101598
200,1.096399
225,1.075008
250,1.112856


TrainOutput(global_step=3103, training_loss=1.0215570885175433, metrics={'train_runtime': 2559.7978, 'train_samples_per_second': 9.696, 'train_steps_per_second': 1.212, 'total_flos': 2.0264311749240422e+17, 'train_loss': 1.0215570885175433, 'epoch': 1.0})

In [52]:
trainer.save_model("./pr-review-qwen-final")
tokenizer.save_pretrained("./pr-review-qwen-final")

('./pr-review-qwen-final/tokenizer_config.json',
 './pr-review-qwen-final/chat_template.jinja',
 './pr-review-qwen-final/tokenizer.json')

In [53]:
import os
os.listdir("./pr-review-qwen-final")

['README.md',
 'adapter_model.safetensors',
 'adapter_config.json',
 'training_args.bin',
 'chat_template.jinja',
 'tokenizer_config.json',
 'tokenizer.json']

In [56]:
!zip -r pr-review-qwen-final.zip pr-review-qwen-final

  adding: pr-review-qwen-final/ (stored 0%)
  adding: pr-review-qwen-final/README.md (deflated 65%)
  adding: pr-review-qwen-final/adapter_model.safetensors (deflated 7%)
  adding: pr-review-qwen-final/adapter_config.json (deflated 59%)
  adding: pr-review-qwen-final/training_args.bin (deflated 53%)
  adding: pr-review-qwen-final/chat_template.jinja (deflated 71%)
  adding: pr-review-qwen-final/tokenizer_config.json (deflated 60%)
  adding: pr-review-qwen-final/tokenizer.json (deflated 81%)


In [59]:
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./pr-review-qwen-final"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [60]:
trainer.state.log_history[-5:]

[{'loss': 0.9413743591308594,
  'grad_norm': 0.3365451395511627,
  'learning_rate': 5.091846600064454e-06,
  'epoch': 0.9748630357718338,
  'step': 3025},
 {'loss': 0.9694690704345703,
  'grad_norm': 0.38601943850517273,
  'learning_rate': 3.4805027392845635e-06,
  'epoch': 0.9829197550757331,
  'step': 3050},
 {'loss': 0.9240720367431641,
  'grad_norm': 0.35114040970802307,
  'learning_rate': 1.8691588785046728e-06,
  'epoch': 0.9909764743796327,
  'step': 3075},
 {'loss': 0.938282470703125,
  'grad_norm': 0.42800912261009216,
  'learning_rate': 2.5781501772478253e-07,
  'epoch': 0.999033193683532,
  'step': 3100},
 {'train_runtime': 2559.7978,
  'train_samples_per_second': 9.696,
  'train_steps_per_second': 1.212,
  'total_flos': 2.0264311749240422e+17,
  'train_loss': 1.0215570885175433,
  'epoch': 1.0,
  'step': 3103}]

In [61]:
trainer.state.global_step

3103

In [62]:
import os

print(os.listdir("./pr-review-qwen-final"))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'training_args.bin', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


In [63]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype="auto",
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "./pr-review-qwen-final"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [64]:
prompt = """
Review the following pull request and provide detailed feedback.

PR Title:
Add JWT authentication middleware and protect user endpoints

PR Description:
This PR introduces JWT authentication for the API.
...
"""

In [65]:
messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=512
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user

Review the following pull request and provide detailed feedback.

PR Title:
Add JWT authentication middleware and protect user endpoints

PR Description:
This PR introduces JWT authentication for the API.
...

assistant
To review this pull request (PR), I would need to see the full context of the changes, including any relevant code snippets or documentation updates. However, based on the provided information, here is my assessment:

1. **Purpose and Context**:
   - The PR aims to add JWT authentication middleware to the API, which is a common practice for securing APIs. This is a good approach as it ensures that only authenticated users can access certain endpoints.

2. **Code Changes**:
   - The PR includes additions related to JWT authentication in various parts of the codebase. It seems to be well-structured with clear comments explaining each step, such as setting up the environment variables, creati

In [66]:
print(dataset[0]["instruction"][:3000])
print("=" * 100)
print(dataset[0]["output"][:3000])

Here is a python code snippet (Some lines may be omitted):

```python

import sys
import threading
import signal
import array
import queue
import time
import os
from os import getpid

from traceback import format_exc

from . import connection
from .context import reduction, get_spawning_popen, ProcessError
from . import pool
from . import process
from . import util
from . import get_context
try:
    from . import shared_memory
    HAS_SHMEM = True
    __all__.append(SharedMemoryManager)
except ImportError:
    HAS_SHMEM = False

#
# Register some things for pickling
#

def reduce_array(a):
    return array.array, (a.typecode, a.tobytes())
reduction.register(array.array, reduce_array)

view_types = [type(getattr({}, name)()) for name in ('items','keys','values')]
if view_types[0] is not list:       # only needed in Py3.0
    def rebuild_as_list(obj):
        return list, (list(obj),)
    for view_type in view_types:
        reduction.register(view_type, rebuild_as_list)

#
# Type for id

In [67]:
sample = dataset[0]["instruction"]

messages = [
    {"role": "user", "content": sample}
]

In [68]:
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False,
    temperature=None
)

In [69]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

model = PeftModel.from_pretrained(
    base_model,
    "ToastCoder/pr-review-qwen"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at 'ToastCoder/pr-review-qwen'